# 🎬 Test đếm trên VIDEO THẬT — dây chuyền nhiều sản phẩm + query suite + lưu video

Nhiều video dây chuyền (kiện hàng, hệ thống chuyền đang chạy, dây chuyền rộng…),
bộ **query suite phong phú** (màu/phụ kiện/hành động/khó/tiếng Việt), người tách
**vạch (vào/ra) + vùng**, và **lưu video output** có vẽ vạch/vùng + box + số đếm.


## 1) Tải code + cài thư viện


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!pip install -q ultralytics 'supervision>=0.21' opencv-python-headless


## 2) ✅ KIỂM TRA tải video (nhanh, không cần model)
Chạy trước để biết nguồn nào tải được. Video ❌ → báo mình ID để đổi nguồn.


In [ ]:
!python run_scenarios.py --download-only


## 3) Xem catalog + toàn bộ query suite


In [ ]:
!python run_scenarios.py --list
import recognition.video_catalog as vc
for t,groups in vc.QUERY_SUITES.items():
    print(f'\n=== {t.upper()} — {sum(len(v) for v in groups.values())} query ===')
    for g,qs in groups.items(): print(f'  [{g}] ' + ' | '.join(qs))


## 4) 🚗 Đếm XE + lưu video output


In [ ]:
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir /kaggle/working/scen_out


## 5) 🚶 Đếm NGƯỜI — cắt VẠCH (vào/ra) + đếm VÙNG + lưu video


In [ ]:
!python run_scenarios.py --task people --max-frames 300 --save-dir /kaggle/working/scen_out


## 6) 📦 Đếm chai trên chuyền (YOLO, nhanh) + lưu video


In [ ]:
!python run_scenarios.py --task conveyor --only milk --max-frames 300 --save-dir /kaggle/working/scen_out


## 7) 📦🧠 Đếm SẢN PHẨM dây chuyền + BỘ QUERY SUITE (open-vocab)
Chạy trên video 'kiện hàng chạy trên chuyền' (nhiều sản phẩm). LocateAnything +
auto-pin transformers + attn sdpa. **Chậm** — giảm `--max-frames` cho nhanh.


In [ ]:
# Suite sản phẩm (chai/hộp/trạng thái/tiếng Việt…) trên video kiện hàng:
!python run_scenarios.py --task conveyor --only packages --suite --max-frames 50 --save-dir /kaggle/working/scen_out


In [ ]:
# Hoặc tất cả video dây chuyền, mỗi video các query gợi ý sẵn:
# !python run_scenarios.py --task conveyor --all-queries --max-frames 50 --save-dir /kaggle/working/scen_out


## 8) 🎥 Xem / tải video output


In [ ]:
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('/kaggle/working/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
if vids:
    src = vids[0]; dst = '/kaggle/working/preview_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    print('Xem:', src); display(Video(dst, embed=True, width=700))


---
### Ghi chú
- **Cell 2** kiểm tra tải trước — video dây chuyền dùng endpoint tải chính thức của Pexels.
- `--suite`: nhiều trường hợp query phân nhóm (cột **nhóm** trong scorecard).
- Video output: vàng=vạch, xanh mờ=vùng, xanh dương=box+#id, banner số đếm.
- Thêm/đổi video hoặc query: sửa `recognition/video_catalog.py`.
